# Lab 3.1 · Hosted, local or neither

Day 3 · Session 1 (Which data may go where) · 25 min.

Before the first request of a project, every table gets a level and a destination. You apply Hudhud's
assistant policy to eight tables and two aggregates, prove one decision on the data, and write the
guard that runs before any prompt leaves the notebook.

All data in this course is synthetic and describes nobody. In this lab you treat it as if it were real.

**AI policy for this lab:** AI-off. The decisions are the analyst's, under the policy.

**Tasks**
- [ ] 1 · Read the assistant policy and the table catalogue (5 min)
- [ ] 2 · Classify each table and choose its destination (8 min)
- [ ] 3 · Find the personal data in the complaint texts (4 min)
- [ ] 4 · Write the guard and test it (4 min)
- [ ] 5 · Write the verification note (4 min)
- [ ] ✱ Stretch · Build an aggregate of complaints that may go to the hosted assistant

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-206.git"
    if not Path("/content/ds-206").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-206"], check=True)
    os.chdir("/content/ds-206/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import pandas as pd
import hudhud as U

pd.set_option("display.width", 160)
con = U.con()  # the first call builds the synthetic tables, once (about 20 seconds)

## 1 · Read the assistant policy and the table catalogue

The policy has eight rules, P-1 to P-8. The catalogue lists each table's columns with the kind of
information each column holds, as recorded by the table's owner.

In [ ]:
print(U.load("docs/assistant_policy.md"))
catalog = U.load("catalog.json")
for item in catalog:
    kinds = sorted(set(item["columns"].values()) - {"operational"})
    print(f"{item['name']:36s} owner: {item['owner']:17s} published: {item['published']!s:5s} other than operational: {', '.join(kinds) or 'nothing'}")

## 2 · Classify each table and choose its destination

The kinds of column are: `operational`, `commercial`, `employment`, `contact`, `name`,
`location of a person`, `free text by a person` and `government id`.

✏️ **Exercise** · Write the two rules as functions.
`classify(item)` returns `(level, personal)`: `personal` is `True` when any column is of a kind that
identifies or describes a person (every kind except `operational` and `commercial`); `level` is
`"Public"` for a published table, and for an aggregate with no personal column and no cell under 20
(P-3), otherwise `"Restricted"` (P-1, P-2).
`destination(item)` returns `"neither"` for a table with a government identity number (P-7), `"hosted"`
for a Public one (P-4), otherwise `"local"` (P-6).

In [ ]:
PERSONAL_KINDS = {"employment", "contact", "name", "location of a person", "free text by a person", "government id"}


def classify(item: dict) -> tuple[str, bool]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


def destination(item: dict) -> str:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
# Check, with a safety net: the cells below still run when the exercise is not done yet
try:
    handling = pd.DataFrame([{"table": i["name"], "level": classify(i)[0], "personal_data": classify(i)[1],
                              "destination": destination(i)} for i in catalog])
except NotImplementedError:
    handling = pd.DataFrame(U.load("reference/handling.json"))
U.keep("handling.json", handling.to_dict("records"))
print(handling.to_string(index=False))
by_dest = handling.groupby("destination")["table"].apply(list).to_dict()
U.check(by_dest["hosted"] == ["hubs", "agg_on_time_by_city_month"], "rows of one table and one aggregate may reach the hosted assistant")
U.check(by_dest["neither"] == ["drivers"], "the drivers table goes to no assistant")
U.check(len(by_dest["local"]) == 7, "seven items stay with the local assistant")
out_dir = U.ART / "lab_3_1"
out_dir.mkdir(parents=True, exist_ok=True)
handling.to_csv(out_dir / "data_handling.csv", index=False)

**Look at** the two aggregates. One is Public and may leave. The other has a district column and cells of
four complaints: an aggregate is not safe because it is an aggregate. And look at `deliveries`: the
course's main table is Restricted. Until now `U.ask` has sent its **schema** only (P-5), never a row.

## 3 · Find the personal data in the complaint texts

The catalogue says `complaints.text` is free text by a person. Prove it without reading a single text:
count the texts that carry a mobile number.

✏️ **Exercise** · Return the share of complaint texts that contain a Saudi mobile number: `05` followed
by eight digits (`regexp_matches(text, '05[0-9]{8}')`). Return a number; print no text.

In [ ]:
def phone_share() -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
# Check, with a safety net
try:
    share = phone_share()
except NotImplementedError:
    share = U.load("reference/lab_3_1_share.json")["share"]
U.keep("lab_3_1_share.json", {"share": round(share, 4)})
n = int(U.one(con, "SELECT COUNT(*) FROM complaints"))
print(f"{share:.1%} of {n:,} complaint texts carry a mobile number: about {share * n:,.0f} texts.")
print(f"The 500 rows that Lab 2.3's summary read would hold about {share * 500:.0f} of them.")
U.check(0.25 < share < 0.40, "about a third of the texts carry a phone number")

**Look at** what a pattern can and cannot do. It finds numbers. Names are in the same texts, and no
pattern lists every name. Masking free text lowers the risk; it does not change the table's level.

## 4 · Write the guard and test it

A policy that lives in a document is followed on good days. The guard runs before every request.

✏️ **Exercise** · `guard(tables, to, what)` raises `PermissionError` with the rule's number when the
request breaks the policy, and returns `True` otherwise. `to` is `"hosted"` or `"local"`; `what` is
`"schema"` or `"rows"`. Use the `destination` column of your handling table: the schema of any table may
go anywhere (P-5); rows go to the hosted assistant only from a `hosted` table (P-4), to the local one
from a `hosted` or `local` table (P-6), and from a `neither` table nowhere (P-7).

In [ ]:
DEST = dict(zip(handling.table, handling.destination))


def guard(tables: list[str], to: str, what: str) -> bool:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
# Check, with a safety net
def blocked(tables, to, what):
    try:
        guard(tables, to, what)
    except PermissionError as e:
        return str(e)
    except NotImplementedError:
        ref = U.load("reference/lab_3_1_guard.json")
        return ref.get(f"{','.join(tables)}|{to}|{what}", "")
    return ""


cases = [(["deliveries", "customers"], "hosted", "schema"), (["hubs"], "hosted", "rows"), (["complaints"], "hosted", "rows"),
         (["complaints"], "local", "rows"), (["drivers"], "local", "rows"), (["agg_on_time_by_city_month"], "hosted", "rows")]
results = {f"{','.join(t)}|{to}|{what}": blocked(t, to, what) for t, to, what in cases}
U.keep("lab_3_1_guard.json", results)
for k, v in results.items():
    print(f"{k:45s} {'BLOCKED  ' + v if v else 'allowed'}")
U.check([bool(v) for v in results.values()] == [False, False, True, False, True, False], "two of the six requests are blocked")
U.check("P-4" in results["complaints|hosted|rows"] and "P-7" in results["drivers|local|rows"], "each refusal names its rule")
print("\nLab 2.3's summary request, 500 complaint texts to a hosted assistant:", results["complaints|hosted|rows"])

## 5 · Write the verification note

Nothing was produced by an assistant in this lab. The note records the decision and how it was checked.

✏️ **Exercise** · Fill the three parts.

In [ ]:
def my_note():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    parts = my_note()
except NotImplementedError:
    parts = U.load("reference/notes.json")["3.1"]
note = U.note("3.1", **parts)

## ✱ Stretch

Build an aggregate of complaints that could be classified Public under P-3: no personal column, no cell
under 20. Which grouping columns can you keep? Who would have to approve it?

## Check yourself

1. `deliveries` is Restricted, and the labs sent requests about it to a hosted model. Which rule allowed that?
2. Why does the drivers table go to no assistant, when a local one never leaves the company?
3. A colleague masks the phone numbers in the complaint texts. May the texts go to the hosted assistant now?

<details><summary>Answers</summary>

1. P-5: only table and column names were sent. No row, sample or example value left the notebook.
2. P-7: no analysis needs a government identity number, so the safest handling is that no model reads
   it. The local assistant gets a view without the column.
3. No. The texts still hold names and what a person wrote; the table stays Restricted. Masking lowers
   the harm of a leak, not the level.

</details>

## Carry forward

`artifacts/lab_3_1/data_handling.csv` is the data-handling table the capstone asks for. Lab 3.4 loads a
supplied copy.